# 02 - Text Annotation & Normalization

**Project:** Hassaniya Arabic Text-To-Speech System Using Transfer Learning  
**Author:** Mohamed Salem Ebnou Echvagha Oubeid (C34613)  
**Module:** NLP Dialects — Master M1 AI  
**Date:** June 2026

---

## Objective

Text annotation is a critical step in any TTS pipeline. Before feeding text to a
speech model, we must:

1. **Normalize** the text (standardize character forms)
2. **Clean** the text (remove noise, fix encoding issues)
3. **Tokenize** at character level (TTS models operate on characters or phonemes)
4. **Build a vocabulary** (map characters to integer IDs)

### Why is this important for Hassaniya?

Hassaniya Arabic has **no standardized orthography**. The same word can be written
in multiple ways. For example, the letter Alef can appear as: ا, أ, إ, آ.
We need to normalize these variants so the TTS model sees consistent input.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

print('Setup complete')

In [ ]:
# Load dataset
df = pd.read_parquet('../audios_dataset.parquet')
print(f'Loaded {len(df)} samples')
print(f'\nFirst 5 transcriptions:')
for i in range(5):
    print(f'  [{i}] {df["transcription"].iloc[i]}')

## 2. Text Normalization

### What is normalization?

Normalization reduces the number of unique characters by mapping variants
to a canonical form. This is especially important for Arabic because:

| Original | Normalized | Reason |
|----------|-----------|--------|
| أ, إ, آ, ٱ | ا | All forms of Alef → base Alef |
| ؤ | و | Hamza on Waw → plain Waw |
| ئ | ي | Hamza on Ya → plain Ya |
| ة | ه | Ta Marbuta → Ha |
| ى | ي | Alef Maqsura → Ya |

We also **remove diacritics** (tashkeel) since our dataset doesn't consistently
use them, and they would create data sparsity.

In [ ]:
class HassaniyaTextProcessor:
    """Text processor tailored for Hassaniya Arabic dialect."""
    
    # Regex to match Arabic diacritical marks (tashkeel)
    ARABIC_DIACRITICS = re.compile(r'[\u064B-\u0670\u06DF]')
    
    # Character normalization mapping
    CHAR_NORMALIZATIONS = {
        '\u0622': '\u0627',  # آ → ا
        '\u0623': '\u0627',  # أ → ا
        '\u0625': '\u0627',  # إ → ا
        '\u0671': '\u0627',  # ٱ → ا
        '\u0624': '\u0648',  # ؤ → و
        '\u0626': '\u064A',  # ئ → ي
        '\u0629': '\u0647',  # ة → ه
        '\u0649': '\u064A',  # ى → ي
        '\u0679': '\u062A',  # ٹ → ت
        '\u06A4': '\u0641',  # ڤ → ف
        '\u06AF': '\u0643',  # گ → ك
    }
    
    def normalize(self, text):
        """Apply full normalization pipeline."""
        # Step 1: Unicode NFKC normalization
        text = unicodedata.normalize('NFKC', text)
        # Step 2: Remove diacritics
        text = self.ARABIC_DIACRITICS.sub('', text)
        # Step 3: Normalize character variants
        for src, dst in self.CHAR_NORMALIZATIONS.items():
            text = text.replace(src, dst)
        # Step 4: Remove non-Arabic characters (keep Arabic + spaces + basic punctuation)
        text = re.sub(r'[^\u0600-\u06FF\s.,\u060C\u061B:\u061F!\u061F\-]', '', text)
        # Step 5: Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def char_tokenize(self, text):
        """Character-level tokenization with space markers."""
        tokens = []
        for ch in text:
            if ch == ' ':
                tokens.append('<space>')
            else:
                tokens.append(ch)
        return tokens
    
    def build_vocab(self, texts):
        """Build character vocabulary from a list of texts."""
        chars = set()
        for text in texts:
            normalized = self.normalize(text)
            chars.update(normalized)
        
        # Special tokens first, then sorted characters
        vocab = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<space>': 3}
        for i, ch in enumerate(sorted(chars - {' '}), start=4):
            vocab[ch] = i
        return vocab
    
    def text_to_ids(self, text, vocab):
        """Convert text to sequence of integer IDs."""
        normalized = self.normalize(text)
        tokens = self.char_tokenize(normalized)
        return [vocab.get(t, 0) for t in tokens]


processor = HassaniyaTextProcessor()
print('Processor initialized')

In [ ]:
# Demonstrate normalization
print('=== Normalization Examples ===')
print()
for i in range(min(8, len(df))):
    original = df['transcription'].iloc[i]
    normalized = processor.normalize(original)
    changed = original != normalized
    marker = ' [CHANGED]' if changed else ''
    print(f'Original:   {original}')
    print(f'Normalized: {normalized}{marker}')
    print()

In [ ]:
# Apply normalization to entire dataset
df['normalized_text'] = df['transcription'].apply(processor.normalize)

# Count how many texts changed
changed_count = (df['transcription'] != df['normalized_text']).sum()
print(f'Texts modified by normalization: {changed_count} / {len(df)} ({100*changed_count/len(df):.1f}%)')

# Compare character counts before and after
original_chars = set(''.join(df['transcription'].tolist())) - {' '}
normalized_chars = set(''.join(df['normalized_text'].tolist())) - {' '}

print(f'\nUnique characters before normalization: {len(original_chars)}')
print(f'Unique characters after normalization:  {len(normalized_chars)}')
print(f'Characters removed/merged: {len(original_chars) - len(normalized_chars)}')

## 3. Character-Level Tokenization

### Why character-level?

TTS models typically work at the **character level** or **phoneme level**:

- **Character-level**: Each Arabic letter becomes a token. Simple and works well for Arabic.
- **Phoneme-level**: Each sound unit becomes a token. More accurate but requires a phoneme dictionary.

For our MVP, we use **character-level tokenization** because:
1. No Hassaniya phoneme dictionary exists
2. Arabic script is largely phonetic — letters map closely to sounds
3. Simpler to implement with our time constraints

In [ ]:
# Build vocabulary
vocab = processor.build_vocab(df['transcription'].tolist())

print(f'Vocabulary size: {len(vocab)}')
print(f'\nVocabulary mapping:')
for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
    display_token = token if token.startswith('<') else f'  {token}  '
    print(f'  {idx:3d} → {display_token}')

In [ ]:
# Demonstrate tokenization
print('=== Tokenization Examples ===')
print()
for i in range(3):
    text = df['transcription'].iloc[i]
    normalized = processor.normalize(text)
    tokens = processor.char_tokenize(normalized)
    ids = processor.text_to_ids(text, vocab)
    
    print(f'Text:     {text}')
    print(f'Tokens:   {tokens}')
    print(f'IDs:      {ids}')
    print(f'Length:   {len(ids)} tokens')
    print()

In [ ]:
# Visualize token length distribution
token_lengths = [len(processor.text_to_ids(t, vocab)) for t in df['transcription']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(token_lengths, bins=30, color='#E91E63', edgecolor='white', alpha=0.8)
ax.set_xlabel('Token Sequence Length')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Token Sequence Lengths After Normalization')
ax.axvline(np.mean(token_lengths), color='red', linestyle='--', 
           label=f'Mean: {np.mean(token_lengths):.0f} tokens')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean token length: {np.mean(token_lengths):.1f}')
print(f'Max token length: {max(token_lengths)}')

## 4. Create Annotated Dataset

We now create the final annotated dataset with:
- Unique ID for each sample
- Original transcription
- Normalized text
- Token sequence
- Token length

In [ ]:
# Create annotated dataset
annotated = pd.DataFrame({
    'id': [f'hassaniya_{i:04d}' for i in range(len(df))],
    'original_text': df['transcription'],
    'normalized_text': df['normalized_text'],
    'token_ids': [str(processor.text_to_ids(t, vocab)) for t in df['transcription']],
    'token_length': token_lengths,
    'text_length': df['transcription'].str.len(),
})

# Save annotated dataset
annotated.to_csv('../data/metadata.csv', index=False, encoding='utf-8')

print(f'Annotated dataset saved: {len(annotated)} samples')
print(f'\nSample entries:')
print(annotated[['id', 'normalized_text', 'token_length']].head(10).to_string())

In [ ]:
# Save vocabulary for TTS model
import json

vocab_path = '../data/vocab.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f'Vocabulary saved to {vocab_path}')
print(f'Vocabulary size: {len(vocab)} tokens')

## 5. Annotation Quality Analysis

In [ ]:
# Summary statistics
print('=' * 60)
print('     ANNOTATION SUMMARY')
print('=' * 60)
print(f'  Total samples annotated:    {len(annotated)}')
print(f'  Vocabulary size:            {len(vocab)} characters')
print(f'  Special tokens:             4 (pad, sos, eos, space)')
print(f'  Arabic characters:          {len(vocab) - 4}')
print(f'  Texts modified by norm:     {changed_count}')
print(f'  Mean token length:          {np.mean(token_lengths):.1f}')
print(f'  Max token length:           {max(token_lengths)}')
print('=' * 60)
print()
print('Annotation pipeline: Raw text → Normalize → Tokenize → ID sequence')
print('Ready for preprocessing (Notebook 03)')

## Conclusion

In this notebook, we built a complete text annotation pipeline for Hassaniya Arabic:

1. **Normalization**: Standardized Alef forms, removed diacritics, cleaned punctuation
2. **Tokenization**: Character-level tokenization with space markers
3. **Vocabulary**: Built a mapping of characters → integer IDs
4. **Annotated dataset**: Saved with original, normalized, and tokenized text

**Key insight**: Normalization reduced our character set, making it easier for
the TTS model to learn consistent mappings from text to speech.

**Next step**: Notebook 03 — Audio preprocessing and feature extraction.